<a href="https://colab.research.google.com/github/BWMIN-Hub/SR_practice/blob/main/notebooks/01_srcnn_x3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SRCNN ×3 — The Beginning of Deep Learning-based Super-Resolution

A blurred satellite image at 10 m, made three times finer — 3.33 m.


## 1. Data

In [ ]:
import sys, urllib.request

LIB = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main/lib'
for m in ['sr_utils.py', 'srcnn_arch.py', 'srcnn_imgproc.py', 'srcnn_models.py']:
    urllib.request.urlretrieve(f'{LIB}/{m}', m)
    sys.modules.pop(m[:-3], None)     # drop a stale copy if one is already imported

from sr_utils import *

show_data()        # 2 validation patches + 2 test areas

## 2. Training

**This is only here to show that the code runs.**  16 images, one epoch.
The results below come from weights trained on the full dataset.

The model is small enough to run without a GPU.  `srcnn_pairs` upscales the LR three
times with bicubic and keeps only the Y channel as the input; the target is the Y
channel of the HR.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from srcnn_models import build_srcnn, srcnn_pairs

N_TRAIN, EPOCHS, BATCH = 16, 1, 4
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

lo, hi = zip(*[pair('training', s) for s in list_split('training')[:N_TRAIN]])
x, t = srcnn_pairs(lo, hi)        # input: Y of the bicubic-upscaled LR,  target: Y of the HR
loader = DataLoader(TensorDataset(x, t), batch_size=BATCH, shuffle=True)

net = build_srcnn().to(dev).train()
opt = torch.optim.Adam(net.parameters(), 2e-4, betas=(0.9, 0.99))
crit = nn.MSELoss()

print(f'SRCNN  {sum(p.numel() for p in net.parameters())/1e3:.1f}K')
for ep in range(1, EPOCHS + 1):
    tot = 0.0
    for xb, tb in loader:
        loss = crit(net(xb.to(dev)), tb.to(dev))
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item()
    print(f'epoch {ep}/{EPOCHS}   MSE {tot/len(loader):.6f}')

## 3. Results

In [ ]:
MODEL = f'{BASE}/models/01_srcnn_x3'

from srcnn_models import load_srcnn, srcnn_upscale

net = load_srcnn(fetch(f'{MODEL}/checkpoints/srcnn_x3.pth', 'srcnn_x3.pth'))
print(f'SRCNN  {sum(p.numel() for p in net.parameters())/1e3:.1f}K')

upscale = lambda lr: srcnn_upscale(net, lr)

show_results(upscale, 'SRCNN', center=[(67, 370), (313, 36)])

## 4. Final test

A real Sentinel-2 capture, with no ground truth.  Judged by eye.


In [ ]:
show_test(upscale, 'SRCNN', center=(800, 800), size=70)